In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils

logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

import torch
import transformers

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

/disk/u/arnab/miniconda3/envs/connection/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-11-14 18:01:05 __main__ INFO     torch.__version__='2.9.0+cu128', torch.version.cuda='12.8'
2025-11-14 18:01:05 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=8, torch.cuda.get_device_name()='NVIDIA A100-SXM4-80GB'
2025-11-14 18:01:05 __main__ INFO     transformers.__version__='4.57.1'


## Loading the LM

In [3]:
from src.utils.training_utils import get_device_map

model_key = "meta-llama/Llama-3.2-3B"
# model_key = "meta-llama/Llama-3.1-8B-Instruct"
# model_key = "meta-llama/Llama-3.3-70B-Instruct"
# model_key = "meta-llama/Llama-3.1-405B-Instruct"

# model_key = "google/gemma-2-9b-it"
# model_key = "google/gemma-2-27b-it"

# model_key = "openai/gpt-oss-20b"

# model_key = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

# model_key = "allenai/OLMo-2-1124-7B-Instruct"
# model_key = "allenai/OLMo-7B-0424-hf"

# model_key = "Qwen/Qwen2-7B"
# model_key = "Qwen/Qwen2.5-14B-Instruct"
# model_key = "Qwen/Qwen2.5-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "Qwen/Qwen3-1.7B"
# model_key = "Qwen/Qwen3-4B"
# model_key = "Qwen/Qwen3-8B"
# model_key = "Qwen/Qwen3-14B"
# model_key = "Qwen/Qwen3-32B"

# device_map = get_device_map(model_key, 30, n_gpus=8)
# device_map

2025-11-14 18:01:10 git.cmd DEBUG    Popen(['git', 'version'], cwd=/disk/u/arnab/Codes/Projects/filter/notebooks, stdin=None, shell=False, universal_newlines=False)
2025-11-14 18:01:10 git.cmd DEBUG    Popen(['git', 'version'], cwd=/disk/u/arnab/Codes/Projects/filter/notebooks, stdin=None, shell=False, universal_newlines=False)


In [4]:
from src.models import ModelandTokenizer

# from transformers import BitsAndBytesConfig

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    # device_map=device_map,
    device_map="auto",
    # quantization_config = BitsAndBytesConfig(
    #     # load_in_4bit=True
    #     load_in_8bit=True
    # )
    attn_implementation="eager",
)

2025-11-14 18:01:16 src.models WARNING  meta-llama/Llama-3.2-3B not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2025-11-14 18:01:16 urllib3.connectionpool DEBUG    Starting new HTTPS connection (1): huggingface.co:443


2025-11-14 18:01:16 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /meta-llama/Llama-3.2-3B/resolve/main/config.json HTTP/1.1" 200 0
2025-11-14 18:01:16 filelock DEBUG    Attempting to acquire lock 140689914578960 on /disk/u/models/.locks/models--meta-llama--Llama-3.2-3B/47d4a5aa69cdef91a53b77f5c5583647a578ca0e.lock
2025-11-14 18:01:16 filelock DEBUG    Lock 140689914578960 acquired on /disk/u/models/.locks/models--meta-llama--Llama-3.2-3B/47d4a5aa69cdef91a53b77f5c5583647a578ca0e.lock
2025-11-14 18:01:16 urllib3.connectionpool DEBUG    https://huggingface.co:443 "GET /meta-llama/Llama-3.2-3B/resolve/main/config.json HTTP/1.1" 200 844
2025-11-14 18:01:16 filelock DEBUG    Attempting to release lock 140689914578960 on /disk/u/models/.locks/models--meta-llama--Llama-3.2-3B/47d4a5aa69cdef91a53b77f5c5583647a578ca0e.lock
2025-11-14 18:01:16 filelock DEBUG    Lock 140689914578960 released on /disk/u/models/.locks/models--meta-llama--Llama-3.2-3B/47d4a5aa69cdef91a53b77f5c5583

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

2025-11-14 18:01:17 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /meta-llama/Llama-3.2-3B/resolve/13afe5124825b4f3751f836b40dafda64c1ed062/model-00002-of-00002.safetensors HTTP/1.1" 302 0
2025-11-14 18:01:17 filelock DEBUG    Attempting to acquire lock 140689912869840 on /disk/u/models/.locks/models--meta-llama--Llama-3.2-3B/4719a04514ec2f060240711b7c33ab21187cac730ecaba3040b7a0fd95a9cefb.lock
2025-11-14 18:01:17 filelock DEBUG    Lock 140689912869840 acquired on /disk/u/models/.locks/models--meta-llama--Llama-3.2-3B/4719a04514ec2f060240711b7c33ab21187cac730ecaba3040b7a0fd95a9cefb.lock
2025-11-14 18:01:17 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /meta-llama/Llama-3.2-3B/resolve/13afe5124825b4f3751f836b40dafda64c1ed062/model-00001-of-00002.safetensors HTTP/1.1" 302 0
2025-11-14 18:01:17 filelock DEBUG    Attempting to acquire lock 140689912774864 on /disk/u/models/.locks/models--meta-llama--Llama-3.2-3B/584d8d3e3f82f7964955174dfe5e3b1cf117a9d8

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.74it/s]

2025-11-14 18:01:38 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /meta-llama/Llama-3.2-3B/resolve/main/generation_config.json HTTP/1.1" 200 0
2025-11-14 18:01:38 filelock DEBUG    Attempting to acquire lock 140688031392080 on /disk/u/models/.locks/models--meta-llama--Llama-3.2-3B/2d73a6863086ff9d491c28e49df9fb697cd92c2b.lock
2025-11-14 18:01:38 filelock DEBUG    Lock 140688031392080 acquired on /disk/u/models/.locks/models--meta-llama--Llama-3.2-3B/2d73a6863086ff9d491c28e49df9fb697cd92c2b.lock
2025-11-14 18:01:38 urllib3.connectionpool DEBUG    https://huggingface.co:443 "GET /meta-llama/Llama-3.2-3B/resolve/main/generation_config.json HTTP/1.1" 200 185
2025-11-14 18:01:38 filelock DEBUG    Attempting to release lock 140688031392080 on /disk/u/models/.locks/models--meta-llama--Llama-3.2-3B/2d73a6863086ff9d491c28e49df9fb697cd92c2b.lock
2025-11-14 18:01:38 filelock DEBUG    Lock 140688031392080 released on /disk/u/models/.locks/models--meta-llama--Llama-3.2-3B/2d73a68

2025-11-14 18:01:39 src.models INFO     loaded model <meta-llama/Llama-3.2-3B> | size: 6127.834 MB | dtype: torch.bfloat16 | device: cuda:0


## Saving the selection data
> For baseline evaluation. So that every LM is evaluated on the same set of data.

In [139]:
from src.selection.data import (
    SelectOneTask,
    SelectFirstTask,
    SelectLastTask,
    YesNoTask,
    CountingTask,
)
from typing import Literal

##########################################################
prompt_template_idx = 3  # try out different templates
option_style: Literal["single_line", "numbered"] = "single_line"
n_distractors = (
    5  # number of distractors. total options = n_distractors + 1 for SingleOne task
)
##########################################################

symantic_type = "objects"
# symantic_type = "profession"
# symantic_type = "nationality"

TASK_CLS = CountingTask

select_task = TASK_CLS.load(
    path=os.path.join(env_utils.DEFAULT_DATA_DIR, "selection", f"{symantic_type}.json")
)
select_task.categories

['fruit',
 'vehicle',
 'furniture',
 'animal',
 'music instrument',
 'clothing',
 'electronics',
 'sport equipment',
 'kitchen appliance',
 'vegetable',
 'building',
 'office supply',
 'bathroom item',
 'flower',
 'tree',
 'jewelry']

In [157]:
sample = select_task.get_random_sample(
    mt=mt,
    option_style=option_style,
    prompt_template_idx=prompt_template_idx,
    category="fruit",
    filter_by_lm_prediction=False,
)

print(sample.prompt(), ">>")
print(f'"{mt.tokenizer.decode([sample.ans_token_id])}"')

predict_next_token(
    inputs=sample.prompt(),
    mt=mt,
)

Items: Bike, Apartment, Elephant, Mango, Banana
How many fruits are in this list?
Answer: >>
" Two"


[[PredictedToken(token=' ', prob=0.41796875, logit=21.5, token_id=220, metadata=None),
  PredictedToken(token=' There', prob=0.287109375, logit=21.125, token_id=2684, metadata=None),
  PredictedToken(token=' Two', prob=0.2236328125, logit=20.875, token_id=9220, metadata=None),
  PredictedToken(token=' Three', prob=0.0235595703125, logit=18.625, token_id=14853, metadata=None),
  PredictedToken(token=' TWO', prob=0.00982666015625, logit=17.75, token_id=47358, metadata=None)]]

In [158]:
import random
random.randint(1,3)

2

In [159]:
from tqdm.auto import tqdm

################################################################################################
LIMIT = 1024
N_DISTRACTORS = 5
DS_ROOT = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR, "selection/baseline", select_task.task_name
)
################################################################################################

os.makedirs(DS_ROOT, exist_ok=True)

evaluation_samples = []
for _ in tqdm(range(LIMIT)):
    sample = select_task.get_random_sample(
        mt=mt,
        option_style=option_style,
        prompt_template_idx=prompt_template_idx,
        n_distractors=N_DISTRACTORS,
        n_options=random.randint(1, 3),
        filter_by_lm_prediction=False,
    )
    evaluation_samples.append(sample)

with open(os.path.join(DS_ROOT, f"{symantic_type}.json"), "w") as f:
    json.dump(
        [sample.to_dict() for sample in evaluation_samples],
        f,
        indent=4,
    )

  0%|          | 0/1024 [00:00<?, ?it/s]

100%|██████████| 1024/1024 [00:17<00:00, 58.63it/s]


## Load Evaluation Samples

In [20]:
from src.selection.data import (
    SelectOneTask,
    SelectFirstTask,
    SelectLastTask,
    YesNoTask,
    CountingTask,
)
from typing import Literal
from src.selection.data import SelectionSample, YesNoSample, CountingSample
from src.selection.utils import get_first_token_id
from src.selection.data import MCQify_sample, COUNT_STR_MAP

##########################################################
prompt_template_idx = 3  # try out different templates
option_style: Literal["single_line", "numbered"] = "single_line"
symantic_type = "objects"
# symantic_type = "profession"
# symantic_type = "nationality"
##########################################################

TASK_CLS = YesNoTask
select_task = TASK_CLS.load(
    path=os.path.join(env_utils.DEFAULT_DATA_DIR, "selection", f"{symantic_type}.json")
)

DS_ROOT = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR, "selection/baseline", select_task.task_name
)

with open(os.path.join(DS_ROOT, f"{symantic_type}.json"), "r") as f:
    raw_samples = json.load(f)

if TASK_CLS == YesNoTask:
    evaluation_samples = [YesNoSample.from_dict(d) for d in raw_samples]
elif TASK_CLS == CountingTask:
    evaluation_samples = [CountingSample.from_dict(d) for d in raw_samples]
else:
    evaluation_samples = [SelectionSample.from_dict(d) for d in raw_samples]

prompt_template = select_task.prompt_templates[prompt_template_idx]

for idx in range(len(evaluation_samples)):
    evaluation_samples[idx].prompt_template = prompt_template
    # evaluation_samples[idx].option_style = option_style
    if isinstance(evaluation_samples[idx], SelectionSample):
        evaluation_samples[idx].ans_token_id = get_first_token_id(
            name=evaluation_samples[idx].answer, tokenizer=mt.tokenizer, prefix=" "
        )
    elif isinstance(evaluation_samples[idx], CountingSample):
        count_str = COUNT_STR_MAP[evaluation_samples[idx].count]
        evaluation_samples[idx].ans_token_id = get_first_token_id(
            name=count_str, tokenizer=mt.tokenizer, prefix=" "
        )
    elif isinstance(evaluation_samples[idx], YesNoSample):
        yes_mode = evaluation_samples[idx].yes
        evaluation_samples[idx].ans_token_id = get_first_token_id(
            "Yes" if yes_mode else "No", tokenizer=mt.tokenizer, prefix=" "
        )
    # evaluation_samples[idx] = MCQify_sample(
    #     sample=evaluation_samples[idx], tokenizer=mt.tokenizer
    # )

sample = evaluation_samples[15]
print(sample.prompt(), ">>", f'"{mt.tokenizer.decode(sample.ans_token_id)}"')

Items: Soap, Surfboard, Potato, Racket, Football, Yoga mat
Do you see a sport equipment in the list above?
Answer: >> " Yes"


In [21]:
from src.selection.utils import get_first_token_id, verify_correct_option
from src.selection.data import get_options_for_answer

result = verify_correct_option(
    mt=mt,
    target=sample.ans_token_id,
    options=get_options_for_answer(sample),
    input=sample.prompt(),
    k=10,
)
result

(True,
 [PredictedToken(token=' Yes', prob=0.6015625, logit=19.25, token_id=7566, metadata=None),
  PredictedToken(token=' R', prob=0.09228515625, logit=17.375, token_id=432, metadata=None),
  PredictedToken(token=' Football', prob=0.049560546875, logit=16.75, token_id=21424, metadata=None),
  PredictedToken(token=' Surf', prob=0.049560546875, logit=16.75, token_id=65197, metadata=None),
  PredictedToken(token=' No', prob=0.043701171875, logit=16.625, token_id=2360, metadata=None),
  PredictedToken(token=' The', prob=0.033935546875, logit=16.375, token_id=578, metadata=None),
  PredictedToken(token=' yes', prob=0.0159912109375, logit=15.625, token_id=10035, metadata=None),
  PredictedToken(token=' A', prob=0.01251220703125, logit=15.375, token_id=362, metadata=None),
  PredictedToken(token=' YES', prob=0.01104736328125, logit=15.25, token_id=14410, metadata=None),
  PredictedToken(token=' ', prob=0.009765625, logit=15.125, token_id=220, metadata=None)],
 OrderedDict([(7566,
           

In [22]:
from tqdm import tqdm

results = []
for sample in tqdm(evaluation_samples):
    is_correct, pred, track = verify_correct_option(
        mt=mt,
        target=sample.ans_token_id,
        options=get_options_for_answer(sample),
        input=sample.prompt(),
    )
    results.append(
        {
            "sample": sample,
            "is_correct": is_correct,
            "predicted_option": pred,
            "track": track,
        }
    )

100%|██████████| 1024/1024 [00:47<00:00, 21.55it/s]


In [23]:
import numpy as np

ranks = []
logits = []
for result in results:
    sample = result["sample"]
    cur_rank = result["track"][sample.ans_token_id][0]
    ranks.append(cur_rank)
    logits.append(result["track"][sample.ans_token_id][1].logit)

n_correct = sum([1 for result in results if result["is_correct"]])
accuracy = n_correct / len(results)

ranks = np.array(ranks)
ranks_avg = ranks.mean()
ranks_std = ranks.std()

logits = np.array(logits)
logits_avg = logits.mean()
logits_std = logits.std()

print(
    f"Accuracy: {accuracy*100:.2f}% ({n_correct}/{len(results)}) | Avg. Rank: {ranks_avg:.2f} ± {ranks_std:.2f} | Avg. Logit: {logits_avg:.2f} ± {logits_std:.2f}"
)

Accuracy: 94.63% (969/1024) | Avg. Rank: 1.34 ± 0.63 | Avg. Logit: 18.01 ± 0.85


In [24]:
print(sample.prompt())

Items: Hospital, Skyscraper, Hairdryer, Potato, Oven, Tomato
Do you see a tree in the list above?
Answer:


In [15]:
failed_cases = [result for result in results if not result["is_correct"]]

In [22]:
sample = failed_cases[26]["sample"]
print(sample.prompt())

Options: Lion, Elm, Toaster, Watch, Pendant, Brooch, Socks, Calculator.
What is the last jewelry in this list above?
Answer:


In [23]:
from src.functional import predict_next_token

predict_next_token(
    inputs=sample.prompt(),
    mt=mt,
)

[[PredictedToken(token=' Pendant', prob=0.498046875, logit=20.375, token_id=65651, metadata=None),
  PredictedToken(token=' Bro', prob=0.439453125, logit=20.25, token_id=4723, metadata=None),
  PredictedToken(token='Pendant', prob=0.01324462890625, logit=16.75, token_id=141484, metadata=None),
  PredictedToken(token=' **', prob=0.01324462890625, logit=16.75, token_id=5231, metadata=None),
  PredictedToken(token='  ', prob=0.01324462890625, logit=16.75, token_id=139, metadata=None)]]

In [24]:
sample.ans_token_id, mt.tokenizer.decode(sample.ans_token_id)

(4723, ' Bro')